In [1]:
# 加载所需要的包
%matplotlib inline

# scitnific computing and plotting
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

# HDDM related packages
import pymc as pm
import hddm
import kabuki
import arviz as az
print("The current HDDM version is: ", hddm.__version__)
print("The current kabuki version is: ", kabuki.__version__)
print("The current PyMC version is: ", pm.__version__)
print("The current ArviZ version is: ", az.__version__)

import warnings
warnings.filterwarnings('ignore') 

The current HDDM version is:  1.0.1RC
The current kabuki version is:  0.6.5RC4
The current PyMC version is:  2.3.8
The current ArviZ version is:  0.15.1


## 加载数据

In [ ]:
# 读取数据并进行预处理，response=choice
data_motion = hddm.load_csv('../../../3_Data/exp1/CleanData/data_motion.csv')
data_color = hddm.load_csv('../../../3_Data/exp1/CleanData/data_color.csv')

# 提取出不同任务的数据
data_motion_match = data_motion[data_motion['part'] == 'match_RDK'] 
data_motion_rdk = data_motion[data_motion['part'] == 'RDK']
data_color_match = data_color[data_color['part'] == 'match_RDK'] 
data_color_rdk = data_color[data_color['part'] == 'RDK']

data_motion_match = data_motion_match.assign(
    correct=data_motion_match['correct'].astype(int),
    response=data_motion_match['response'].map({'f': 1, 'j': 0, 'arrowleft': 1, 'arrowright': 0}),
    stim2=data_motion_match['isMatch'].map({'match': 1, 'mismatch': 0}), # 这里进行了修改（z flip需要）
    stim=data_motion_match['isMatch'].map({'match': 1, 'mismatch': -1})
)

data_motion_rdk = data_motion_rdk.assign(
    correct=data_motion_rdk['correct'].astype(int),
    response=data_motion_rdk['response'].map({'f': 1, 'j': 0, 'arrowleft': 1, 'arrowright': 0}),
    coherent_direction=data_motion_rdk['coherent_direction'].replace(180, 1), # 左为1
    stim = data_motion_rdk['coherent_direction'].map({180:1, 0:0})

)

data_color_match = data_color_match.assign(
    correct=data_color_match['correct'].astype(int),
    response=data_color_match['response'].map({'f': 1, 'j': 0}),
    stim2=data_color_match['isMatch'].map({'match': 1, 'mismatch': 0}),
    stim=data_color_match['isMatch'].map({'match': 1, 'mismatch': -1})
)

data_color_rdk = data_color_rdk.assign(
    correct=data_color_rdk['correct'].astype(int),
    response=data_color_rdk['response'].map({'f': 1, 'j': 0, 'd': 1, 'k': 0}),
    stim=data_color_rdk['dot_color_final'].map({'["hsl(225, 50%, 50%)","hsl(0, 50%, 50%)"]':0, '["hsl(0, 50%, 50%)","hsl(225, 50%, 50%)"]':1}), # 红为1，蓝为0
)

# 新填choice列，方便后面转换编码方式
data_motion_rdk['choice'] = data_motion_rdk['response']
data_color_rdk['choice'] = data_color_rdk['response']
data_motion_match['choice'] = data_motion_match['response']
data_color_match['choice'] = data_color_match['response']


# data_match
data_color_match

,subj_idx,rt,correct,coherence,part,response,dot_color_final,isMatch,association,difficulty,stim,choice
0,100,1.397,0,0.0,match_RDK,0,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",match,self,3,1,0
1,100,1.224,1,0.0,match_RDK,0,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",mismatch,other,1,-1,0
2,100,1.123,0,0.0,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",mismatch,other,2,-1,1
3,100,1.599,1,0.0,match_RDK,0,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",mismatch,other,4,-1,0
4,100,1.227,1,0.0,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",match,other,3,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...
40801,99,1.472,1,0.0,match_RDK,1,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",match,other,4,1,1
40802,99,1.408,0,0.0,match_RDK,0,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",match,self,3,1,0
40803,99,1.186,1,0.0,match_RDK,0,"[""hsl(225, 50%, 50%)"",""hsl(0, 50%, 50%)""]",mismatch,other,2,-1,0
40804,99,1.504,1,0.0,match_RDK,0,"[""hsl(0, 50%, 50%)"",""hsl(225, 50%, 50%)""]",mismatch,self,2,-1,0


## 运动组-匹配任务

采用回归模型，设置参数翻转
- m0：基础模型，仅包含a,v,t,z和dc四个参数
- m1：v随matchness和association变化，z随association变化
- m2：v随matchness和association变化，dc随associaition变化，z随association变化
- m3：v随matchness，association和difficulty变化，z随association变化
- m4：v随matchness，association和difficulty变化，dc随associaition变化，z随association变化

注：m1和m3中不包含dc

### m0：仅包含a,v,t,z和dc

**choice coding + v flip，有dc**

In [ ]:
data_motion_match['response'] = data_motion_match['choice']
    
# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + stim", 'link_func': lambda x: x}
] 
 
match_task_motion_m0 = hddm.HDDMRegressor(
    data_motion_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    group_only_regressors=False
)

match_task_motion_m0_idata = match_task_motion_m0.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_motion_m0.db', db = 'pickle')
match_task_motion_m0.save('match_task_motion_m0.hddm')

In [ ]:
summary = az.summary(match_task_motion_m0_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### m1：v随matchness和association变化，z随association变化

**accuracy coding + z flip，无dc**

In [ ]:
data_motion_match['response'] = data_motion_match['correct']

# 定义链接函数
def z_link_func(x, data=data_motion_match):
    stim = data.stim2.loc[x.index] # 数据中的stim2编码为1和0
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*isMatch", 'link_func': lambda x: x}
] 
 
match_task_motion_m1 = hddm.HDDMRegressor(
    data_motion_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_motion_m1_idata = match_task_motion_m1.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_motion_m1.db', db = 'pickle')
match_task_motion_m1.save('match_task_motion_m1.hddm')

In [4]:
summary = az.summary(match_task_motion_m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                mean     sd  hdi_3%  hdi_97%   
a                                              2.075  0.027   2.027    2.127  \
a_std                                          0.218  0.021   0.180    0.258   
t                                              0.865  0.029   0.812    0.920   
t_std                                          0.241  0.022   0.202    0.282   
z_Intercept(other)                             0.517  0.002   0.512    0.521   
z_Intercept(self)                              0.521  0.002   0.517    0.525   
z_Intercept_std                                0.012  0.002   0.010    0.015   
v_Intercept                                    0.471  0.032   0.410    0.529   
v_Intercept_std                                0.232  0.025   0.188    0.282   
v_association[T.self]                          0.047  0.050  -0.049    0.139   
v_association[T.self]_std                      0.370  0.036   0.307    0.438   
v_isMatch[T.mismatch]                   

### m2: v随matchness和association变化，dc随associaition变化，z随association变化

**choice coding + v flip，有dc**

In [ ]:
data_motion_match['response'] = data_motion_match['choice']

# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + association*stim", 'link_func': lambda x: x}
] 
 
match_task_motion_m2 = hddm.HDDMRegressor(
    data_motion_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_motion_m2_idata = match_task_motion_m2.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_motion_m2.db', db = 'pickle')
match_task_motion_m2.save('match_task_motion_m2.hddm')


In [6]:
summary = az.summary(match_task_motion_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                 mean     sd  hdi_3%  hdi_97%  mcse_mean   
a                               2.074  0.027   2.022    2.124      0.000  \
a_std                           0.217  0.021   0.179    0.255      0.000   
t                               0.872  0.028   0.819    0.923      0.000   
t_std                           0.241  0.022   0.201    0.282      0.000   
z(other)                        0.534  0.005   0.525    0.542      0.000   
z(self)                         0.566  0.005   0.557    0.574      0.000   
z_std                           0.107  0.013   0.084    0.132      0.001   
v_Intercept                    -0.043  0.019  -0.076   -0.006      0.001   
v_Intercept_std                 0.113  0.015   0.086    0.140      0.000   
v_association[T.self]          -0.048  0.021  -0.085   -0.006      0.001   
v_association[T.self]_std       0.080  0.025   0.033    0.127      0.002   
v_stim                          0.479  0.031   0.422    0.539      0.000   
v_stim_std  

### m3: v随matchness，association和difficulty变化，z随association变化

**accuracy coding + z flip，无dc**

In [ ]:
data_motion_match['response'] = data_motion_match['correct']

# 定义链接函数
def z_link_func(x, data=data_motion_match):
    stim = data.stim2.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*isMatch*difficulty", 'link_func': lambda x: x}
] 
 
match_task_motion_m3 = hddm.HDDMRegressor(
    data_motion_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_motion_m3_idata = match_task_motion_m3.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_motion_m3.db', db = 'pickle')
match_task_motion_m3.save('match_task_motion_m3.hddm')


In [ ]:
summary = az.summary(match_task_motion_m3_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                     mean     sd  hdi_3%   
a                                                   2.095  0.028   2.042  \
a_std                                               0.225  0.021   0.187   
t                                                   0.863  0.028   0.809   
t_std                                               0.241  0.022   0.200   
z_Intercept(other)                                  0.517  0.002   0.513   
z_Intercept(self)                                   0.522  0.002   0.518   
z_Intercept_std                                     0.013  0.001   0.010   
v_Intercept                                         0.910  0.047   0.823   
v_Intercept_std                                     0.253  0.030   0.201   
v_association[T.self]                               0.235  0.071   0.106   
v_association[T.self]_std                           0.403  0.044   0.322   
v_isMatch[T.mismatch]                              -0.016  0.057  -0.119   
v_isMatch[T.

### m4: v随matchness，association和difficulty变化，dc随associaition变化，z随association变化

**choice coding + v flip，有dc**

In [ ]:
data_motion_match['response'] = data_motion_match['choice']

# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + association*stim*difficulty", 'link_func': lambda x: x}
] 
 
match_task_motion_m4 = hddm.HDDMRegressor(
    data_motion_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_motion_m4_idata = match_task_motion_m4.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_motion_m4.db', db = 'pickle')
match_task_motion_m4.save('match_task_motion_m4.hddm')


In [ ]:
summary = az.summary(match_task_motion_m4_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

[-----------------95%----------------  ] 1915 of 2000 complete in 16364.7 sec
[-----------------95%----------------  ] 1916 of 2000 complete in 16374.4 sec
[-----------------95%----------------  ] 1917 of 2000 complete in 16384.6 sec
[-----------------95%----------------  ] 1918 of 2000 complete in 16392.7 sec
[-----------------95%----------------  ] 1919 of 2000 complete in 16401.4 sec
[-----------------96%----------------  ] 1920 of 2000 complete in 16409.6 sec
[-----------------96%----------------  ] 1921 of 2000 complete in 16418.1 sec
[-----------------96%----------------  ] 1922 of 2000 complete in 16426.3 sec
[-----------------96%----------------  ] 1923 of 2000 complete in 16434.5 sec
[-----------------96%----------------  ] 1924 of 2000 complete in 16443.1 sec
[-----------------96%----------------  ] 1925 of 2000 complete in 16451.8 sec
[-----------------96%----------------  ] 1926 of 2000 complete in 16460.4 sec
[-----------------96%----------------  ] 1927 of 2000 complete i

In [11]:
summary = az.summary(match_task_motion_m4_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                            mean     sd  hdi_3%  hdi_97%   
a                                          2.092  0.028   2.037    2.144  \
a_std                                      0.222  0.021   0.182    0.261   
t                                          0.872  0.029   0.815    0.923   
t_std                                      0.240  0.022   0.202    0.282   
z(other)                                   0.534  0.005   0.525    0.543   
z(self)                                    0.564  0.005   0.556    0.574   
z_std                                      0.108  0.013   0.083    0.131   
v_Intercept                                0.005  0.029  -0.047    0.059   
v_Intercept_std                            0.094  0.019   0.060    0.131   
v_association[T.self]                      0.003  0.036  -0.068    0.067   
v_association[T.self]_std                  0.043  0.032   0.001    0.098   
v_stim                                     0.864  0.040   0.786    0.937   
v_stim_std  

### 模型比较

In [ ]:
# match_task_motion_m1 = hddm.load('match_task_motion_m1.hddm')
# match_task_motion_m2 = hddm.load('match_task_motion_m2.hddm')
# match_task_motion_m3 = hddm.load('match_task_motion_m3.hddm')
# match_task_motion_m4 = hddm.load('match_task_motion_m4.hddm')

In [ ]:
DIC_dict = {
    "m0": match_task_motion_m0.dic,
    "m1": match_task_motion_m1.dic,
    "m2": match_task_motion_m2.dic,
    "m3": match_task_motion_m3.dic,
    "m4": match_task_motion_m4.dic
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,72619.251582
m2,m2,72483.313578
m3,m3,71722.486826
m4,m4,71600.449166


## 颜色组-匹配任务

采用回归模型，设置参数翻转
- m0：基础模型
- m1：v随matchness和association变化，z随association变化
- m2：v随matchness和association变化，dc随associaition变化，z随association变化
- m3：v随matchness，association和difficulty变化，z随association变化
- m4：v随matchness，association和difficulty变化，dc随associaition变化，z随association变化

### m0：仅包含a, v, t, z和dc

**choice coding + v flip，有dc**

In [ ]:
data_color_match['response'] = data_color_match['choice']
    
# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + stim", 'link_func': lambda x: x}
] 
 
match_task_color_m0 = hddm.HDDMRegressor(
    data_color_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    group_only_regressors=False
)

match_task_color_m0_idata = match_task_color_m0.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_color_m0.db', db = 'pickle')
match_task_color_m0.save('match_task_color_m0.hddm')

In [ ]:
summary = az.summary(match_task_color_m0_idata)
# 只筛选出不含"_subj."的参数行
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### m1：v随matchness和association变化，z随association变化

**accuracy coding + z flip，无dc**

In [ ]:
data_color_match['response'] = data_color_match['correct']

# 定义链接函数
def z_link_func(x, data=data_color_match):
    stim = data.stim2.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*isMatch", 'link_func': lambda x: x}
] 
 
match_task_color_m1 = hddm.HDDMRegressor(
    data_color_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_color_m1_idata = match_task_color_m1.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_color_m1.db', db = 'pickle')
match_task_color_m1.save('match_task_color_m1.hddm')

In [4]:
summary = az.summary(match_task_color_m1_idata)
# 只筛选出不含"_subj."的参数行
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                                mean     sd  hdi_3%  hdi_97%   
a                                              1.966  0.027   1.915    2.018  \
a_std                                          0.223  0.021   0.183    0.259   
t                                              0.681  0.016   0.651    0.711   
t_std                                          0.133  0.012   0.111    0.156   
z_Intercept(other)                             0.519  0.002   0.514    0.523   
z_Intercept(self)                              0.528  0.002   0.523    0.532   
z_Intercept_std                                0.015  0.001   0.012    0.017   
v_Intercept                                    0.703  0.041   0.627    0.781   
v_Intercept_std                                0.321  0.031   0.265    0.379   
v_association[T.self]                          0.039  0.079  -0.110    0.186   
v_association[T.self]_std                      0.629  0.056   0.521    0.732   
v_isMatch[T.mismatch]                   

### m2: v随matchness和association变化，dc随associaition变化，z随association变化

**choice coding + v flip，有dc**

In [ ]:
data_color_match['response'] = data_color_match['choice']

# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + association*stim", 'link_func': lambda x: x}
] 
 
match_task_color_m2 = hddm.HDDMRegressor(
    data_color_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_color_m2_idata = match_task_color_m2.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_color_m2.db', db = 'pickle')
match_task_color_m2.save('match_task_color_m2.hddm')

In [6]:
summary = az.summary(match_task_color_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

[-----------------95%----------------  ] 1914 of 2000 complete in 17849.4 sec
[-----------------95%----------------  ] 1915 of 2000 complete in 17858.5 sec
[-----------------95%----------------  ] 1916 of 2000 complete in 17867.0 sec
[-----------------95%----------------  ] 1917 of 2000 complete in 17875.3 sec
[-----------------95%----------------  ] 1918 of 2000 complete in 17883.8 sec
[-----------------95%----------------  ] 1919 of 2000 complete in 17892.4 sec
[-----------------96%----------------  ] 1920 of 2000 complete in 17900.9 sec
[-----------------96%----------------  ] 1921 of 2000 complete in 17909.1 sec
[-----------------96%----------------  ] 1922 of 2000 complete in 17918.5 sec
[-----------------96%----------------  ] 1923 of 2000 complete in 17926.9 sec
[-----------------96%----------------  ] 1924 of 2000 complete in 17935.2 sec
[-----------------96%----------------  ] 1925 of 2000 complete in 17943.7 sec
[-----------------96%----------------  ] 1926 of 2000 complete i

### m3: v随matchness，association和difficulty变化，z随association变化

**accuracy coding + z flip, 无dc**

In [ ]:
data_color_match['response'] = data_color_match['correct']

# 定义链接函数
def z_link_func(x, data=data_color_match):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*isMatch*difficulty", 'link_func': lambda x: x}
] 
 
match_task_color_m3 = hddm.HDDMRegressor(
    data_color_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_color_m3_idata = match_task_color_m3.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_color_m3.db', db = 'pickle')
match_task_color_m3.save('match_task_color_m3.hddm')

In [8]:
summary = az.summary(match_task_color_m3_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

[-----------------95%----------------  ] 1907 of 2000 complete in 54863.7 sec
[-----------------95%----------------  ] 1908 of 2000 complete in 54885.3 sec
[-----------------95%----------------  ] 1909 of 2000 complete in 54907.1 sec
[-----------------95%----------------  ] 1910 of 2000 complete in 54930.6 sec
[-----------------95%----------------  ] 1911 of 2000 complete in 54952.8 sec
[-----------------95%----------------  ] 1912 of 2000 complete in 54974.1 sec
[-----------------95%----------------  ] 1913 of 2000 complete in 54995.4 sec
[-----------------95%----------------  ] 1914 of 2000 complete in 55016.8 sec
[-----------------95%----------------  ] 1915 of 2000 complete in 55039.0 sec
[-----------------95%----------------  ] 1916 of 2000 complete in 55060.2 sec
[-----------------95%----------------  ] 1917 of 2000 complete in 55081.4 sec
[-----------------95%----------------  ] 1918 of 2000 complete in 55102.8 sec
[-----------------95%----------------  ] 1919 of 2000 complete i

### m4: v随matchness，association和difficulty变化，dc随associaition变化，z随association变化

**choice coding + v flip，有dc**

In [ ]:
data_color_match['response'] = data_color_match['choice']

# 构建回归模型
model_reg = [
    {'model': "v ~ 1 + association*stim*difficulty", 'link_func': lambda x: x}
] 
 
match_task_color_m4 = hddm.HDDMRegressor(
    data_color_match,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    depends_on={'z': ['association']},
    group_only_regressors=False
)

match_task_color_m4_idata = match_task_color_m4.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'match_task_color_m4.db', db = 'pickle')
match_task_color_m4.save('match_task_color_m4.hddm')

In [ ]:
summary = az.summary(match_task_color_m4_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                            mean     sd  hdi_3%  hdi_97%
a                                          2.033  0.030   1.905    2.143
a_std                                      0.240  0.023   0.175    0.337
t                                          0.687  0.016   0.631    0.759
t_std                                      0.133  0.012   0.095    0.189
z_std                                      0.137  0.012   0.100    0.177
v_Intercept                               -0.116  0.038  -0.248    0.003
v_Intercept_std                            0.083  0.026   0.014    0.172
v_association[T.self]                      0.186  0.053   0.061    0.329
v_association[T.self]_std                  0.052  0.028   0.002    0.164
v_stim                                     1.843  0.057   1.600    2.100
v_stim_std                                 0.412  0.043   0.297    0.633
v_association[T.self]:stim                -0.016  0.081  -0.332    0.285
v_association[T.self]:stim_std             0.606  0

### 模型比较

In [ ]:
# match_task_color_m1 = hddm.load('match_task_color_m1.hddm')
# match_task_color_m2 = hddm.load('match_task_color_m2.hddm')
# match_task_color_m3 = hddm.load('match_task_color_m3.hddm')
# match_task_color_m4 = hddm.load('match_task_color_m4.hddm')

In [ ]:
DIC_dict = {
    "m0": match_task_color_m0.dic,
    "m1": match_task_color_m1.dic,
    "m2": match_task_color_m2.dic,
    "m3": match_task_color_m3.dic,
    "m4": match_task_color_m4.dic
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,61044.607031
m2,m2,60928.043199
m3,m3,57385.039674
m4,m4,57358.463120


## 运动组-辨别任务
- m0：基础模型
- m1: v随association变化
- m2：v随association和difficulty变化

In [17]:
data_motion_rdk

,subj_idx,rt,correct,coherence,part,response,coherent_direction,isMatch,association,difficulty,stim,choice
347,1,0.802,1,0.12,RDK,1,1,NaN,other,1,1,1
348,1,1.213,1,0.08,RDK,1,1,NaN,other,2,1,1
349,1,0.678,1,0.03,RDK,0,0,NaN,self,4,-1,0
350,1,0.878,1,0.03,RDK,1,1,NaN,other,4,1,1
351,1,0.638,1,0.08,RDK,0,0,NaN,self,2,-1,0
...,...,...,...,...,...,...,...,...,...,...,...,...
38948,9,1.347,1,0.12,RDK,1,1,NaN,other,1,1,1
38949,9,0.773,1,0.09,RDK,1,1,NaN,other,3,1,1
38950,9,0.697,1,0.09,RDK,1,1,NaN,other,2,1,1
38951,9,0.823,1,0.09,RDK,0,0,NaN,self,2,-1,0


### m0

In [ ]:
data_motion_rdk['response'] = data_motion_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_motion_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1", 'link_func': lambda x: x}
] 
 
rdk_task_motion_m0 = hddm.HDDMRegressor(
    data_motion_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_motion_m0_idata = rdk_task_motion_m0.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_motion_m0.db', db = 'pickle')
rdk_task_motion_m0.save('rdk_task_motion_m0.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_motion_m0_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### m1: v随association变化

In [ ]:
data_motion_rdk['response'] = data_motion_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_motion_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association", 'link_func': lambda x: x}
] 
 
rdk_task_motion_m1 = hddm.HDDMRegressor(
    data_motion_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_motion_m1_idata = rdk_task_motion_m1.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_motion_m1.db', db = 'pickle')
rdk_task_motion_m1.save('rdk_task_motion_m1.hddm')

In [21]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_motion_m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   
a                          1.300  0.020   1.261    1.336      0.000    0.000  \
a_std                      0.161  0.016   0.134    0.191      0.000    0.000   
t                          0.488  0.018   0.457    0.522      0.000    0.000   
t_std                      0.147  0.013   0.123    0.172      0.000    0.000   
z_Intercept                0.495  0.003   0.490    0.500      0.000    0.000   
z_Intercept_std            0.021  0.002   0.017    0.026      0.000    0.000   
v_Intercept                0.655  0.044   0.574    0.738      0.001    0.001   
v_Intercept_std            0.313  0.038   0.245    0.386      0.001    0.001   
v_association[T.self]     -0.051  0.061  -0.168    0.062      0.001    0.001   
v_association[T.self]_std  0.427  0.057   0.323    0.535      0.002    0.001   

                           ess_bulk  ess_tail  r_hat  
a                            3444.0    4488.0   1.00  
a_std    

### m2: v随association和difficulty变化

In [ ]:
data_motion_rdk['response'] = data_motion_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_motion_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*difficulty", 'link_func': lambda x: x}
] 
 
rdk_task_motion_m2 = hddm.HDDMRegressor(
    data_motion_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_motion_m2_idata = rdk_task_motion_m2.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_motion_m2.db', db = 'pickle')
rdk_task_motion_m2.save('rdk_task_motion_m2.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_motion_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### 模型比较

In [ ]:
DIC_dict = {
    "m0": rdk_task_motion_m0.dic,
    "m1": rdk_task_motion_m1.dic,
    "m2": rdk_task_motion_m2.dic,
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,15242.165691
m2,m2,14909.011104


## 颜色组辨别任务
- m0：基础模型
- m1: v随association变化
- m2：v随association和difficulty变化

### m0

In [ ]:
data_color_rdk['response'] = data_color_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_color_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1", 'link_func': lambda x: x}
] 
 
rdk_task_color_m0 = hddm.HDDMRegressor(
    data_color_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_color_m0_idata = rdk_task_color_m0.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_color_m0.db', db = 'pickle')
rdk_task_color_m0.save('rdk_task_color_m0.hddm')

In [ ]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_color_m0_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

### m1：v随association变化

In [ ]:
data_color_rdk['response'] = data_color_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_color_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association", 'link_func': lambda x: x}
] 
 
rdk_task_color_m1 = hddm.HDDMRegressor(
    data_color_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_color_m1_idata = rdk_task_color_m1.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_color_m1.db', db = 'pickle')
rdk_task_color_m1.save('rdk_task_color_m1.hddm')

In [32]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_color_m1_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd   
a                          1.329  0.023   1.287    1.373      0.000    0.000  \
a_std                      0.179  0.017   0.148    0.212      0.000    0.000   
t                          0.500  0.015   0.473    0.528      0.000    0.000   
t_std                      0.124  0.012   0.104    0.147      0.000    0.000   
z_Intercept                0.467  0.008   0.451    0.482      0.000    0.000   
z_Intercept_std            0.063  0.006   0.051    0.075      0.000    0.000   
v_Intercept                1.061  0.051   0.964    1.157      0.001    0.001   
v_Intercept_std            0.376  0.044   0.295    0.456      0.001    0.001   
v_association[T.self]     -0.008  0.095  -0.185    0.169      0.002    0.001   
v_association[T.self]_std  0.733  0.078   0.589    0.879      0.002    0.001   

                           ess_bulk  ess_tail  r_hat  
a                            3442.0    3652.0    1.0  
a_std    

### m2

In [ ]:
data_color_rdk['response'] = data_color_rdk['correct']

# 定义链接函数
def z_link_func(x, data=data_color_rdk):
    stim = data.stim.loc[x.index]
    z_flip = (stim * x) + (1 - x) * (1 - stim)
    return z_flip
    
# 构建回归模型
model_reg = [
    {'model': "z ~ 1", 'link_func': z_link_func},
    {'model': "v ~ 1 + association*difficulty", 'link_func': lambda x: x}
] 
 
rdk_task_color_m2 = hddm.HDDMRegressor(
    data_color_rdk,
    model_reg,
    bias=True,
    include=['v', 'a', 't', 'z'],
    # depends_on={'z': ['association']},
    group_only_regressors=False
)

rdk_task_color_m2_idata = rdk_task_color_m2.sample(5000, burn=1000, chains=4, return_infdata=True, dbname = 'rdk_task_color_m2.db', db = 'pickle')
rdk_task_color_m2.save('rdk_task_color_m2.hddm')

In [5]:
# 只筛选出不含"_subj."的参数行
summary = az.summary(rdk_task_color_m2_idata)
group_params = summary[~summary.index.str.contains('_subj.')]
print(group_params)

                                       mean     sd  hdi_3%  hdi_97%   
a                                     1.365  0.024   1.322    1.411  \
a_std                                 0.192  0.018   0.159    0.227   
t                                     0.499  0.015   0.471    0.526   
t_std                                 0.124  0.012   0.103    0.146   
z_Intercept                           0.465  0.008   0.449    0.481   
z_Intercept_std                       0.065  0.007   0.053    0.077   
v_Intercept                           2.396  0.079   2.253    2.546   
v_Intercept_std                       0.435  0.055   0.339    0.539   
v_association[T.self]                -0.021  0.131  -0.262    0.225   
v_association[T.self]_std             0.804  0.088   0.647    0.972   
v_difficulty                         -0.498  0.022  -0.540   -0.457   
v_difficulty_std                      0.074  0.018   0.039    0.108   
v_association[T.self]:difficulty      0.006  0.032  -0.064    0.061   
v_asso

### 模型比较

In [ ]:
# rdk_task_color_m1 = hddm.load('rdk_task_color_m1.hddm')
# rdk_task_color_m2 = hddm.load('rdk_task_color_m2.hddm')

In [ ]:
DIC_dict = {
    "m0": rdk_task_color_m0.dic,
    "m1": rdk_task_color_m1.dic,
    "m2": rdk_task_color_m2.dic,
}

DIC_table = pd.DataFrame.from_dict(DIC_dict, orient="index", columns=["DIC"])
DIC_table["model"] = DIC_table.index
DIC_table = DIC_table[["model", "DIC"]]
DIC_table.sort_values(by=["DIC"], ascending=False)

,model,DIC
m1,m1,12874.968865
m2,m2,11502.421202
